# 02 — Modelagem NLP e Avaliação

Este notebook atende aos itens da Etapa 2 sobre:
- tratamento da base;
- preparação para treinamento;
- bases teóricas do método;
- definição de como a acurácia será calculada.

## Estratégia adotada
- base principal: `ifood-restaurants-february-2021.csv`
- foco textual: `name`, `category`, `tags`
- exclusão de registros com `rating == 0`
- criação de alvo binário:
  - `alta_avaliacao` se `rating >= 4.5`
  - `demais` caso contrário
- vetorização por TF-IDF
- classificação supervisionada com Regressão Logística


In [2]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

ROOT = Path("..").resolve()
DATA_PATH = ROOT / "data" / "raw" / "ifood-restaurants-february-2021.csv"

df = pd.read_csv(DATA_PATH, usecols=["name", "category", "tags", "rating"])
df.head()

,category,name,rating,tags
0,Marmita,Cantina Arte & Sabor,0.0,ADDRESS_PREFORM_TYPE $$ CART::MCHT::100_DELIVE...
1,Açaí,Raruty Açaí Raiz,0.0,ADDRESS_PREFORM_TYPE $$ GUIDED_HELP_TYPE $$ ME...
2,Bebidas,Toma na Kombi,0.0,ADDRESS_PREFORM_TYPE $$ CPGN_USER_DISCOUNT_6_L...
3,Carnes,Churrasquinho do Barriga´s,0.0,ADDRESS_PREFORM_TYPE $$ GUIDED_HELP_TYPE $$ NO...
4,Brasileira,Prime Restaurante,0.0,ADDRESS_PREFORM_TYPE $$ GUIDED_HELP_TYPE $$ NOVO


In [3]:
STOPWORDS = {
    "a", "as", "o", "os", "de", "da", "do", "das", "dos", "e", "em", "no", "na",
    "nos", "nas", "um", "uma", "uns", "umas", "para", "por", "com", "sem", "ao",
    "aos", "à", "às", "que", "se", "ou", "como", "mais", "menos"
}

def strip_accents(text):
    text = unicodedata.normalize("NFKD", str(text))
    return "".join(ch for ch in text if not unicodedata.combining(ch))

def normalize_text(text):
    text = str(text).lower()
    text = strip_accents(text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [tok for tok in text.split() if tok not in STOPWORDS and len(tok) > 1]
    return " ".join(tokens)

df = df[df["rating"] > 0].copy()
df["text_input"] = (
    df[["name", "category", "tags"]]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .apply(normalize_text)
)

df = df[df["text_input"].str.len() > 0].copy()
df["target"] = df["rating"].apply(lambda x: "alta_avaliacao" if x >= 4.5 else "demais")

df[["text_input", "target"]].head()

,text_input,target
255,flank steak burger lanches address preform typ...,alta_avaliacao
279,beleleu burger bar casa forte lanches address ...,demais
883,mr sacada variada abr lanche address preform t...,demais
952,sallvattore mediterranea abr pascoa almoco aco...,alta_avaliacao
2053,nippon sushi poke japonesa address preform typ...,demais


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text_input"],
    df["target"],
    test_size=0.2,
    random_state=42,
    stratify=df["target"],
)

pipeline = Pipeline(
    steps=[
        ("tfidf", TfidfVectorizer(min_df=5, ngram_range=(1, 2))),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)

pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)

In [5]:
acc = accuracy_score(y_test, pred)
cm = confusion_matrix(y_test, pred)
report = classification_report(y_test, pred)

print("Acurácia:", round(acc, 4))
print("\nMatriz de confusão:")
print(cm)
print("\nRelatório de classificação:")
print(report)

Acurácia: 0.6277

Matriz de confusão:
[[18827 11441]
 [ 5480  9701]]

Relatório de classificação:
                precision    recall  f1-score   support

alta_avaliacao       0.77      0.62      0.69     30268
        demais       0.46      0.64      0.53     15181

      accuracy                           0.63     45449
     macro avg       0.62      0.63      0.61     45449
  weighted avg       0.67      0.63      0.64     45449



## Interpretação para o relatório técnico

Ao transpor este notebook para o relatório:
- explique por que `rating == 0` foi removido;
- justifique a criação da variável binária;
- descreva por que TF-IDF é adequado para texto;
- descreva por que Regressão Logística foi escolhida;
- apresente a fórmula da acurácia;
- complemente com precisão, recall e F1-score.
